# Tái hiện TransHLA — đánh giá trọng số đã công bố

Bài báo: **TransHLA: a Hybrid Transformer model for HLA-presented epitope detection**
Lu T., Wang X., Nie W., Huo M., Li S. — *GigaScience* 14 (2025), DOI `10.1093/gigascience/giaf008`

Notebook này **không huấn luyện lại**. Nó lấy trọng số tác giả công bố trên HuggingFace
(`SkywalkerLu/TransHLA_I`, `SkywalkerLu/TransHLA_II`) và chạy trên **đúng tập test** tác giả
công bố trong repo gốc, rồi đối chiếu với **Bảng 2** và **Bảng 4** của bài báo.

Đây là bước nên làm trước tiên khi tái hiện: nếu con số khớp, ta biết dữ liệu và quy trình
đánh giá đã đúng, và mọi sai lệch về sau chỉ có thể đến từ quá trình huấn luyện.

| | |
|---|---|
| Runtime cần | GPU (A100 / L4 / T4 đều được) |
| Thời gian | ~10-25 phút cho cả hai lớp HLA trên A100 |
| Tải về | ESM2-650M (~2,6 GB) + 2 checkpoint TransHLA (~2,6 GB mỗi cái) |
| Dữ liệu | clone từ repo gốc, ~250 MB, không cần Drive |

**Runtime → Change runtime type → GPU** trước khi chạy.

## 0. Đồng bộ code từ GitHub

Ô này clone (hoặc `reset --hard` về) đúng bản code trên GitHub, nên `import transhla_eval`
luôn lấy bản bạn vừa push từ VS Code.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/Tienanh209/TransHLA_repro.git"
REPO_DIR = "/content/TransHLA_repro"

if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

## 1. Cài thư viện

`fair-esm` là thư viện ESM2 của Meta — code của mô hình trên HuggingFace gọi trực tiếp vào nó
(`esm.pretrained.esm2_t33_650M_UR50D()`), nên bắt buộc phải có.

In [ ]:
%pip install -q fair-esm "transformers>=4.40" 2>&1 | tail -2

In [ ]:
import torch, platform
print("python     ", platform.python_version())
print("torch      ", torch.__version__)
print("CUDA       ", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    print("\n!!! Chưa bật GPU: Runtime -> Change runtime type -> GPU. "
          "Chạy ở CPU sẽ mất nhiều giờ.")

## 2. Lấy dữ liệu gốc của tác giả

Repo `SkywalkerLuke/TransHLA` chứa sẵn các tập đã chia. Đáng chú ý: số lượng khớp đúng bài báo
— 92.347 epitope HLA-I dương tính và 65.105 epitope HLA-II dương tính trong tập test, nên
không cần dựng lại quy trình sinh mẫu âm (BLAST toàn bộ NCBI nr + CD-HIT) mà bài báo mô tả.

In [ ]:
import transhla_eval as te

official = te.ensure_official_repo("/content/TransHLA_official")
!ls -la {official}/data | head -20

In [ ]:
import pandas as pd

rows = []
for hla, cfg in te.CONFIGS.items():
    for name in cfg.datasets:
        _, info = te.load_dataset(official, cfg, name)
        rows.append({"HLA": hla, "tập": name, **info})
pd.DataFrame(rows)[["HLA", "tập", "file", "rows_raw", "rows_used",
                    "positives", "negatives", "dropped_out_of_length",
                    "n_duplicate_peptides", "n_ambiguous_residue"]]

Các tập `test` dùng nguyên vẹn. Chỉ vài dòng của tập `external` bị loại vì độ dài nằm ngoài
khoảng mô hình được huấn luyện (HLA-I 8-14, HLA-II 13-21) — hàm đệm của tác giả chỉ đệm chứ
không cắt, nên để nguyên sẽ lỗi tensor lệch chiều.

## 3. Chạy trên tập test IEDB — đối chiếu Bảng 2

Đây là phần tốn thời gian nhất: 184.697 + 130.634 peptide đi qua ESM2-650M.
Mỗi peptide chỉ 16-23 token nên vẫn nhanh.

Nếu muốn thử nhanh trước, đặt `SUBSET_N = 20_000`. Để `None` khi chạy thật.

In [ ]:
SUBSET_N = None      # None = toàn bộ tập test
BATCH_SIZE = 256     # A100: 512 cũng được. T4: 128.
AMP = False          # True = autocast fp16, nhanh gần gấp đôi, lệch ~0,01 điểm
OUT_DIR = "/content/results"

model_cache = {}
results = {}

for hla in ["I", "II"]:
    results[(hla, "test")] = te.run(
        hla, dataset="test", official_repo=official, out_dir=OUT_DIR,
        batch_size=BATCH_SIZE, subset_n=SUBSET_N, amp=AMP, model_cache=model_cache,
    )
    r = results[(hla, "test")]
    print(f"HLA-{hla}: {r['seconds']}s, {r['peptides_per_second']} peptide/s")

### Kết quả tái hiện so với bài báo

In [ ]:
for hla in ["I", "II"]:
    r = results[(hla, "test")]
    print(f"\n=== HLA-{hla}, tập test IEDB (n = {r['metrics']['n']:,}) ===")
    display(te.comparison_table(hla, "test", r["metrics"]))

Cột **Lệch** là giá trị tái hiện trừ giá trị bài báo. Lệch dưới 0,5 điểm phần trăm coi như khớp
(khác biệt do phiên bản thư viện, thứ tự batch, và việc bài báo làm tròn tới 2 chữ số).

### Bảng 2 đầy đủ của bài báo, chèn thêm dòng tái hiện

In [ ]:
tab2 = te.PAPER_BASELINES_TABLE2.copy()
extra = []
for hla in ["I", "II"]:
    m = results[(hla, "test")]["metrics"]
    extra.append({
        "Type": f"HLA-{hla}", "Method": "TransHLA (tái hiện)",
        "ACC (%)": round(m["acc"], 2), "F1 (%)": round(m["f1"], 2),
        "Recall (%)": round(m["recall"], 2), "MCC": round(m["mcc"], 2),
        "AUC (%)": round(m["auc"], 2),
    })
tab2 = pd.concat([tab2, pd.DataFrame(extra)], ignore_index=True)
tab2 = tab2.sort_values(["Type", "ACC (%)"], ascending=[True, False])
tab2.to_csv(f"{OUT_DIR}/table2_with_reproduction.csv", index=False)
tab2.reset_index(drop=True)

### Hình 3 — ROC và Precision-Recall

In [ ]:
import numpy as np

for hla in ["I", "II"]:
    pred = pd.read_csv(f"{OUT_DIR}/predictions_hla{hla}_test.csv")
    te.plot_curves(
        pred["label"].to_numpy(), pred["prob"].to_numpy(),
        title=f"HLA-{hla}, tập test IEDB",
        out_path=f"{OUT_DIR}/fig3_hla{hla}.png",
    )

## 4. Neoepitope — đối chiếu Bảng 4 (NEPDB)

Bài báo báo cáo TransHLA đạt **ACC 90,65%**, **recall 93,52%**, **specificity 87,77%** trên
tập neoepitope NEPDB đã kiểm chứng bằng thí nghiệm TCR. Tập này rất nhỏ (278 peptide HLA-I,
22 peptide HLA-II) nên chạy chỉ vài giây.

In [ ]:
for hla in ["I", "II"]:
    results[(hla, "nepdb")] = te.run(
        hla, dataset="nepdb", official_repo=official, out_dir=OUT_DIR,
        batch_size=BATCH_SIZE, amp=AMP, model_cache=model_cache,
    )
    r = results[(hla, "nepdb")]
    print(f"\n=== HLA-{hla}, NEPDB (n = {r['metrics']['n']}) ===")
    display(te.comparison_table(hla, "nepdb", r["metrics"]))

## 5. Tập external — CEDAR, VDJdb, ImmuneCode, dbPepNeo2.0

Các tập này đánh giá khả năng tổng quát hoá: epitope dương tính lấy từ 4 cơ sở dữ liệu khác
IEDB, mẫu âm sinh với tỷ lệ 1:1 (`external_1x`) và 1:4 (`external_4x`). Bài báo trình bày ở
Bảng 3 và Hình 4-6; ở đây chỉ in chỉ số thô để so sánh định tính.

In [ ]:
rows = []
for hla in ["I", "II"]:
    for ds in ["external_1x", "external_4x"]:
        r = te.run(hla, dataset=ds, official_repo=official, out_dir=OUT_DIR,
                   batch_size=BATCH_SIZE, amp=AMP, model_cache=model_cache)
        results[(hla, ds)] = r
        m = r["metrics"]
        rows.append({"HLA": hla, "tập": ds, "n": m["n"],
                     "ACC (%)": round(m["acc"], 2), "F1 (%)": round(m["f1"], 2),
                     "Recall (%)": round(m["recall"], 2),
                     "Specificity (%)": round(m["specificity"], 2),
                     "MCC": round(m["mcc"], 3), "AUC (%)": round(m["auc"], 2)})
external = pd.DataFrame(rows)
external.to_csv(f"{OUT_DIR}/external_results.csv", index=False)
external

## 6. Lưu kết quả

Mặc định mọi thứ nằm ở `/content/results` và mất khi phiên Colab kết thúc. Ô dưới copy sang
Drive nếu bạn muốn giữ.

In [ ]:
SAVE_TO_DRIVE = False   # đổi True để lưu
DRIVE_DIR = "/content/drive/MyDrive/s1156052/EpitopePrediction/results_pretrained"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil, os
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for f in os.listdir(OUT_DIR):
        shutil.copy(f"{OUT_DIR}/{f}", DRIVE_DIR)
    print("đã lưu:", sorted(os.listdir(DRIVE_DIR)))
else:
    !ls -la {OUT_DIR}

## 7. Nếu con số không khớp — kiểm theo thứ tự này

1. **Độ dài đệm.** HLA-I phải đệm tới **16** token, HLA-II tới **23** (= `max_len` 14/21 cộng
   `<cls>` và `<eos>`). Đệm bằng token **1** (`<pad>`). Sai chỗ này thì contact map của ESM2
   lệch chiều và điểm số tụt rõ.
2. **Lẫn mô hình.** `TransHLA_I` chỉ dùng cho peptide 8-14 acid amin, `TransHLA_II` cho 13-21.
   Peptide 13-14 nằm ở cả hai khoảng — đừng lấy mô hình nọ chạy tập kia.
3. **Đầu ra đã là softmax.** `model(tokens)` trả về `(probs, representation)` với `probs[:, 1]`
   là p(epitope) — xem `logits_clsf = F.softmax(...)` cuối `forward` trong `model.py`. Đừng
   softmax lần nữa.
4. **Lọc dữ liệu.** Giữ nguyên tập test: không khử trùng lặp (`train.py` chỉ khử ở tập train),
   không loại peptide chứa `X` (ESM2 có ký tự này trong từ điển). Số dương tính phải đúng
   92.347 (HLA-I) và 65.105 (HLA-II).
5. **`model.eval()`.** Mô hình có BatchNorm và Dropout; quên `eval()` thì kết quả phụ thuộc
   vào thành phần từng batch. `te.load_model` đã gọi sẵn.
6. **fp16.** Nếu đang bật `AMP = True`, thử lại với `False`.

## 8. Bước sau

Khi bảng trên đã khớp, có thể huấn luyện lại từ đầu bằng chính `train.py` của tác giả với
siêu tham số đã công bố: Adam `lr=3e-5`, `weight_decay=0.0025`, batch 64, 100 epoch,
early stopping patience 5, seed 10038, `n_layers=6`, `n_head=8`, `d_model=1280`.

```bash
cd /content/TransHLA_official/model_train_test
python train.py --train_path ../data/HLA_I_epitope_train_shuffle.csv \
                --validation_path ../data/HLA_I_epitope_validation.csv \
                --model_path /content/checkpoints --model_name transhla_I.pt
```

Lưu ý `train.py` gọi ESM2 trong `forward` mỗi batch, mỗi epoch tính lại từ đầu — chậm. Muốn
nhanh thì cache embedding + contact map một lần (đổi lại ~38 GB mỗi lớp HLA ở fp16).